# QAMP Q1b: read role holders kept in plain getters

Top-up to Q1. Rule fixed in `rwa-quantumclock/docs/Q1B_PLAN.md` before any data were fetched.

**Steps**
1. Check `REPO_REF` in Cell 1 (the tag that fixes the plan).
2. *Runtime → Run all*. Paste a **new** Etherscan key when asked (hidden, not saved).
3. It takes a few minutes. If it stops, run Cell 2 again; finished tokens are skipped.
4. `qamp_q1b_data.zip` downloads at the end. Unzip it into `rwa-quantumclock/data/` and commit.
5. Delete the key on etherscan.io/apidashboard.


In [ ]:
# Cell 1: setup (only edit REPO_REF)
REPO_URL = "https://github.com/FahdSaif/RWA-Asset-tokenization.git"
REPO_REF = "quantumclock-q1b-plan"     # tag that fixes the Q1b plan

import subprocess, sys, os
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
sys.path.insert(0, "repo/rwa-quantumclock/src")
commit = subprocess.run(["git", "-C", "repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("plan tag", REPO_REF, "commit", commit[:10])


In [ ]:
# Cell 2: run (nothing to edit; safe to run again)
import datetime as dt, getpass, json, zipfile
from pathlib import Path
import requests
from qamp.extraction import Observer, digest, observe_role_getters, q1b_targets

raw = json.loads(Path("repo/rwa-quantumclock/data/q1_raw.json").read_text())
TARGETS = q1b_targets(raw)
by = {t["address"]: t for t in raw["tokens"]}
print(len(TARGETS), "tokens with role getters")
out = Path("qamp_q1b")
(out / "tokens").mkdir(parents=True, exist_ok=True)

key = getpass.getpass("Paste your Etherscan API key (hidden): ").strip()
def get_json(u):
    try:
        r = requests.get(u, timeout=60); r.raise_for_status(); return r.json()
    except requests.RequestException as e:
        raise RuntimeError(f"network error: {type(e).__name__}") from None   # hide the URL: it contains the key

ob = Observer(get_json, key)
block = ob.block()
for i, a in enumerate(TARGETS, 1):
    f = out / "tokens" / f"{a}.json"
    if f.exists():
        continue
    rec = observe_role_getters(ob, by[a])
    f.write_text(json.dumps(rec))
    holders = sorted({h for h in rec["getters"].values() if h})
    print(f"{i:3d} {a[:10]}... getters={len(rec['getters'])} holders={len(holders)} "
          f"{' '.join(h[:8] for h in holders)} events={len(rec['events'])} errors={len(rec['errors'])}")
del key

records = [json.loads((out / "tokens" / f"{a}.json").read_text()) for a in TARGETS]
retrieved = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
q1b = dict(tokens=records, block=block, retrieved_at_utc=retrieved, plan_ref=REPO_REF, plan_commit=commit)
(out / "q1b_raw.json").write_text(json.dumps(q1b))
all_holders = sorted({h for r in records for h in r["getters"].values() if h})
summary = dict(n_tokens=len(records), distinct_holders=len(all_holders), with_errors=sum(bool(r["errors"]) for r in records),
               block=block, retrieved_at_utc=retrieved, plan_ref=REPO_REF, plan_commit=commit, sha256_raw=digest(q1b))
(out / "q1b_summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
with zipfile.ZipFile("qamp_q1b_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in ("q1b_raw.json", "q1b_summary.json"):
        z.write(out / p, p)
try:
    from google.colab import files; files.download("qamp_q1b_data.zip")
except ImportError:
    print("wrote qamp_q1b_data.zip")
